# Goto response: how the mount follows a goto

A `utility/goto_response.py` run is a fixed sequence of motor-angle gotos (`Polaris:SlewAbsolute` m1/m2/m3), each out
from a base pose and back, with `log_position` on: every control tick (~0.2 s) the driver logs the planned motor angles
(θ_sp), the measured ones (θ_pv), the speed commanded to each motor (ω_op) and the PID terms (PIDLOG), and the script
marks each leg in the log (REPLAYLOG). Several runs (tracking off/on, on battery or plugged in) can share one log.

Per leg and moving motor this compares the goto with the **theoretical minimum**: independent motors, each
accelerating, cruising and decelerating within the planner's limits (`GOTO_RATE_SHARE` of Kv, Ka), the slowest
deciding. The gap splits into how the motors respond (commanded vs measured speed, the response delay), how the
reference is moved (planned trajectory, or the PID chasing it while tracking), overshoot and settling.
The findings are written up in `docs/study_goto_response.md`.

Run the test: `uv run python utility/goto_response.py` (add `--track` to run the gotos while tracking, `--plan` to
check the moves without moving, `--twin` to dry-run in the digital twin).

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import importlib, analyse_helpers as ah
importlib.reload(ah)                                     # module changes without a kernel restart

LOG = os.path.abspath(os.path.join('..', 'logs', 'archive', 'alpaca.jdm_Beta8.0_10_15_Goto_Response_a1.log'))
KV = ah.GR_KV_DEFAULT             # motors' max speed, deg/s: the planner uses GOTO_RATE_SHARE of these
KA = ah.GR_KA_DEFAULT             # acceleration, deg/s^2
EXCLUDE = [(4, 14)]             # (run, case) legs to leave out of the summary: run 4's m3_5_back was stopped by hand
TEMPLATE = 'plotly_dark'
MOTOR_COLOURS = {'M1': 'deepskyblue', 'M2': 'lightgreen', 'M3': 'orange'}

pid, marks, battery = ah.load_goto_response(LOG)
print(f'{len(pid)} PIDLOG ticks, {len(ah.goto_response_runs(marks))} runs, {len(battery)} battery reports')

## 1. The runs

One row per run: tracking, the battery as reported during it (the driver logs a report only when it changes; the
charging flag and level are unreliable around plugging in and out), and the medians over its legs: arrival less the
theoretical minimum (`late`), overshoot, and while tracking the time from within 1' to within the tracking tolerance
(`pid_Kc`/20). Per motor, over the large moves (>= 25 deg): measured / commanded speed and the peak measured speed.

In [ ]:
summary, tables = ah.goto_response_summary(pid, marks, battery, KV, KA, exclude=EXCLUDE)
display(summary.round(2))

## 2. Every leg

`t_min` the theoretical minimum, `t_arrive` within 1' to stay (tracking: of the tracked setpoint), `done` when the
driver reported the goto complete, `behind_ref` how far the motor fell behind its reference, `final` the error at
the end of the hold (idle: within the tolerance `pid_Kc` 0.75' by design).

In [ ]:
RUN = 1
cols = ['case', 'leg', 'motor', 'move_deg', 't_min_s', 't_arrive_s', 'done_s', 'late_s', 'cmd_peak_dps', 'meas_peak_dps',
        'meas_per_cmd', 'lag_s', 'behind_ref_arcmin', 'overshoot_arcmin', 'final_arcmin', 't_tight_s']
display(tables[RUN][cols].round(2))

## 3. Arrival against the theoretical minimum

Each point a leg's moving motor. On the diagonal a goto would be as fast as the motors allow; the vertical distance
is the time lost to overshoot, chasing and settling.

In [ ]:
fig = go.Figure()
for run, df in tables.items():
    if not len(df):
        continue
    d = df.drop_duplicates('case')
    track = summary.set_index('run').loc[run, 'tracking']
    fig.add_scatter(x=d['t_min_s'], y=d['t_arrive_s'], mode='markers', name=f"run {run} ({'tracking' if track else 'idle'})",
                    text=d['leg'], hovertemplate='%{text}<br>min %{x:.1f}s, arrived %{y:.1f}s')
top = max(df['t_arrive_s'].max() for df in tables.values() if len(df))
fig.add_scatter(x=[0, top], y=[0, top], mode='lines', name='theoretical minimum', line=dict(dash='dot', color='grey'))
fig.update_layout(xaxis_title='theoretical minimum (s)', yaxis_title='arrived within 1\' (s)', height=480, template=TEMPLATE)
fig.show()

## 4. One leg in detail

Pick the run with `RUN`, then step through its legs with the slider: per moving motor, the error to the target
(tracking: to the tracked setpoint), the commanded and measured speed, and the PID terms. Idle gotos follow a
planned trajectory (the feed-forward `ff` carries the motion, the PID corrects); gotos while tracking aren't planned -
the reference steps toward the target (up to 12 deg along the sky path) and the PID chases it (`kp`), with `kd`
damping its own command back toward the feed-forward.

In [ ]:
RUN = 4                         # the run to step through
LEG = 0                         # the leg the slider starts on (index into the legs where a motor moved)

run_legs = ah.goto_response_legs(ah.goto_response_runs(marks)[RUN - 1][3])
track = summary.set_index('run').loc[RUN, 'tracking']
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=('error to the target (arcmin)' if not track else 'error to the tracked setpoint (arcmin)',
                                    'commanded (solid) and measured (dotted) speed, deg/s', 'PID terms, deg/s'))
counts, titles, names = [], [], []
for name, t0, t1, target, done in run_legs:
    s = ah.goto_response_leg_series(pid, t0, t1)
    n0 = len(fig.data)
    moving = [a for a in range(3) if abs(target[a] - s[f'm{a + 1}_pv'].iloc[0]) > ah.GR_MOVING_DEG] if len(s) else []
    for a in moving:
        m, M = f'm{a + 1}', f'M{a + 1}'
        c = MOTOR_COLOURS[M]
        err = (s[f'{m}_pv'] - (s[f'{m}_sp'] if track else target[a])) * 60
        fig.add_scatter(x=s['t'], y=err.clip(-300, 300), name=f'{M} error', line=dict(color=c), row=1, col=1)
        fig.add_scatter(x=s['t'], y=s[f'{m}_op'], name=f'{M} ω_op', line=dict(color=c), row=2, col=1)
        fig.add_scatter(x=s['t'], y=s[f'{m}_vel'], name=f'{M} measured', line=dict(color=c, dash='dot'), row=2, col=1)
        for k, dash in (('kp', 'solid'), ('kd', 'dot'), ('ff', 'dash')):
            fig.add_scatter(x=s['t'], y=s[f'{m}_{k}'], name=f'{M} ω_{k}', line=dict(color=c, dash=dash, width=1), row=3, col=1)
    if not moving:
        continue                    # no motor moved (e.g. setup when already at the base pose)
    counts.append(len(fig.data) - n0)
    names.append(name)
    moves = ', '.join(f"M{a + 1} {target[a] - s[f'm{a + 1}_pv'].iloc[0]:+.1f}°" for a in moving)
    titles.append(f'run {RUN} {name}: {moves} - completed after {done:.1f} s')
starts = np.cumsum([0] + counts[:-1])
active = LEG % len(names)
steps = []
for i, (s0, n) in enumerate(zip(starts, counts)):
    vis = [False] * len(fig.data)
    vis[s0:s0 + n] = [True] * n
    steps.append(dict(method='update', label=names[i], args=[{'visible': vis}, {'title': titles[i]}]))
for j, tr in enumerate(fig.data):
    tr.visible = starts[active] <= j < starts[active] + counts[active]
fig.add_hline(y=0, line=dict(color='gray', width=0.8), row=1, col=1)
fig.update_xaxes(title_text='seconds from the goto', row=3, col=1)
fig.update_layout(height=820, template=TEMPLATE, title=titles[active],
                  sliders=[dict(active=active, steps=steps, currentvalue=dict(prefix='leg: '), pad=dict(t=40))])
fig.show()

## 5. What the Kd term does while tracking

With coordinated speed control `ω_kd = -Kd (ω_fb - ω_ff)` acts on the PID's own feedback command, not on a
measured velocity: it damps the command back toward the feed-forward (the tracking rate). A sustained error `e`
then gets `ω_fb = Kp e / (1 + Kd)` - the effective gain is `1/(1 + Kd)` - and the error closes as a first-order
decay with time constant `(1 + Kd)/Kp` (Kd 0.5, 0.5, 0.8: 1.5, 1.5, 1.8 s), lengthened by the response delay and
the `Ke` smoothing. Closing 5 deg to 1' takes `ln(300)` time constants.

In [ ]:
LEGS_AXES = [('m1_5_out', 0), ('m2_5_out', 1), ('m3_5_out', 2), ('all_30_out', 0), ('all_30_out', 2)]
for run in (3, 4):
    print(f'run {run}')
    display(ah.goto_response_kd_effect(pid, ah.goto_response_runs(marks)[run - 1][3], LEGS_AXES).round(2))